# Notebook 23 — Alert-Semantic Lookahead Selection (ASL-Select)

V-C is a valid channel-level alert-semantic diagnostic, but G5 shows that static ranking
does not reliably choose the best recovered pruning set. This notebook tests a stronger,
set-conditional hypothesis.

**ASL-Select** uses V-C only to form a cheap candidate shortlist. It then evaluates the
*marginal deployed semantic risk* of adding each candidate to the current pruning set and
chooses the least damaging addition per realised FLOP. This is a validation-only,
lookahead/approximate-oracle experiment.

The generic idea of lookahead/oracle pruning is not new. The paper may claim novelty only
for the alert-semantic objective, ASVG/AWBIR integration, and the causal benchmark.

Run the shallow architecture first. Enable the depth arm only when the shallow gate is
credible.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src/saber").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError("Set SABER_REPO to the saber-ids-method repository.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO)
print("Device:", DEVICE)
print("Python:", sys.version.split()[0], "|", platform.platform())


In [ ]:
from copy import deepcopy
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset

from src.saber.bridge_ciciot import load_bridge
from src.saber.deep_model import DeepCNN1D
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.adapters import collect_logits
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters

OUT = REPO / "results/saber/23_alert_semantic_lookahead"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_FLOPS = 0.40
FLOP_TOLERANCE = 0.015
CANDIDATE_POOL = 12
AUDIT_PER_CLASS = 128
RUN_SHALLOW = True
RUN_DEEP = False  # enable only after reviewing the shallow result
SEED = 23026

TRAIN_LOADER, VAL_LOADER, TEST_LOADER, SHALLOW, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
graph = pd.read_csv(REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv")
SHALLOW = SHALLOW.to(DEVICE).eval()

deep_path = REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt"
DEEP = None
if RUN_DEEP:
    if not deep_path.exists():
        raise FileNotFoundError(deep_path)
    DEEP = DeepCNN1D(len(CLASS_NAMES))
    DEEP.load_state_dict(torch.load(deep_path, map_location="cpu", weights_only=False)["state_dict"])
    DEEP = DEEP.to(DEVICE).eval()

shallow_scores = pd.read_csv(
    REPO / "results/saber/15_baselines/group_baseline_scores.csv"
).merge(
    pd.read_csv(REPO / "results/saber/16b_score_v2/v2_scores.csv")[["group_id", "v_c"]],
    on="group_id", how="left",
)
deep_scores = (
    pd.read_csv(REPO / "results/saber/20_depth_probe/deep_group_scores.csv")
    if RUN_DEEP else None
)


In [ ]:
# Fixed class-balanced validation audit subset.
Xv, yv = VAL_LOADER.dataset.tensors
rng = np.random.default_rng(SEED)
indices = []
y_np = yv.numpy()
for c in range(len(CLASS_NAMES)):
    pool = np.flatnonzero(y_np == c)
    take = min(AUDIT_PER_CLASS, len(pool))
    indices.extend(rng.choice(pool, size=take, replace=False).tolist())
indices = np.asarray(sorted(indices))
AUDIT_LOADER = DataLoader(TensorDataset(Xv[indices], yv[indices]), batch_size=1024)
EXAMPLE = next(iter(AUDIT_LOADER))[0][:8].to(DEVICE)

def teacher_cache(model):
    logits, labels, _ = collect_logits(model, AUDIT_LOADER, device=DEVICE)
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    return logits, labels, audit

def audit_student(model, teacher_logits, labels):
    logits, observed, _ = collect_logits(model, AUDIT_LOADER, device=DEVICE)
    if not np.array_equal(labels, observed):
        raise RuntimeError("Audit labels changed.")
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        teacher_logits, logits, labels, graph
    )
    audit["awbir"] = float(awbir)
    return audit

def objective(audit, teacher):
    # All terms are validation-only and stated before selection.
    family_loss = max(0.0, teacher["family_macro_f1"] - audit["family_macro_f1"])
    attack_miss_excess = max(
        0.0, audit["attack_to_benign_rate"] - teacher["attack_to_benign_rate"] - 0.01
    )
    false_alert_excess = max(
        0.0, audit["benign_to_attack_rate"] - teacher["benign_to_attack_rate"] - 0.05
    )
    hsr_delta = max(0.0, audit["hsr_balanced_soc"] - teacher["hsr_balanced_soc"])
    return (
        audit["awbir"]
        + 0.50 * hsr_delta
        + 0.25 * family_loss
        + 2.00 * attack_miss_excess
        + 1.00 * false_alert_excess
    )

def build_model(teacher, selected, minimum_width):
    prune_map = {}
    for layer, channel, _ in selected:
        prune_map.setdefault(layer, []).append(channel)
    prune_map = {layer: sorted(channels) for layer, channels in prune_map.items()}
    model, _ = prune_cnn1d_channels(
        teacher, prune_map, EXAMPLE, minimum_remaining_per_layer=minimum_width
    )
    return model.to(DEVICE)


In [ ]:
def run_lookahead(name, teacher, scores, minimum_width):
    teacher_logits, labels, teacher_audit = teacher_cache(teacher)
    dense_flops = profile_forward_flops(teacher, EXAMPLE)["flops_per_item"]
    remaining_by_layer = scores.groupby("module_path")["group_id"].count().to_dict()
    selected = []
    selected_ids = set()
    trace = []
    current_objective = 0.0
    current_realised = 0.0

    static_order = scores.sort_values("v_c", ascending=True)
    iteration = 0
    while current_realised < TARGET_FLOPS:
        iteration += 1
        eligible = []
        for row in static_order.itertuples():
            if row.group_id in selected_ids:
                continue
            layer = str(row.module_path)
            removed_in_layer = sum(1 for l, _, _ in selected if l == layer)
            if remaining_by_layer[layer] - removed_in_layer - 1 < minimum_width:
                continue
            eligible.append((layer, int(row.channel_index), str(row.group_id), float(row.v_c)))
            if len(eligible) >= CANDIDATE_POOL:
                break
        if not eligible:
            raise RuntimeError("No eligible candidate before reaching target.")

        candidates = []
        for layer, channel, gid, v_c in eligible:
            trial_selected = selected + [(layer, channel, gid)]
            trial_model = build_model(teacher, trial_selected, minimum_width)
            realised = 1.0 - profile_forward_flops(
                trial_model, EXAMPLE
            )["flops_per_item"] / dense_flops
            audit = audit_student(trial_model, teacher_logits, labels)
            risk = objective(audit, teacher_audit)
            delta_flops = max(realised - current_realised, 1e-8)
            candidates.append({
                "layer": layer,
                "channel": channel,
                "group_id": gid,
                "v_c": v_c,
                "realised": float(realised),
                "objective": float(risk),
                "marginal_objective_per_flop": float((risk - current_objective) / delta_flops),
                **{key: float(audit[key]) for key in [
                    "awbir", "hsr_balanced_soc", "family_macro_f1",
                    "fine_macro_f1", "attack_to_benign_rate",
                    "benign_to_attack_rate", "ece15",
                ]},
            })
        choice = min(
            candidates,
            key=lambda row: (
                row["marginal_objective_per_flop"],
                row["objective"],
                row["v_c"],
            ),
        )
        selected.append((choice["layer"], choice["channel"], choice["group_id"]))
        selected_ids.add(choice["group_id"])
        current_realised = choice["realised"]
        current_objective = choice["objective"]
        trace.append({"iteration": iteration, **choice})
        if iteration % 10 == 0 or current_realised >= TARGET_FLOPS:
            pd.DataFrame(trace).to_csv(OUT / f"{name}_lookahead_trace.csv", index=False)
            print(name, iteration, "realised=", round(current_realised, 4),
                  "AWBIR=", round(choice["awbir"], 4))

    # Choose the trace prefix nearest the target, not automatically the first overshoot.
    trace_df = pd.DataFrame(trace)
    best_i = int((trace_df["realised"] - TARGET_FLOPS).abs().idxmin())
    chosen_trace = trace_df.iloc[:best_i + 1]
    chosen_ids = chosen_trace["group_id"].tolist()
    chosen = [
        (str(scores.set_index("group_id").loc[gid, "module_path"]),
         int(scores.set_index("group_id").loc[gid, "channel_index"]), gid)
        for gid in chosen_ids
    ]
    final = build_model(teacher, chosen, minimum_width)
    realised = 1.0 - profile_forward_flops(final, EXAMPLE)["flops_per_item"] / dense_flops
    final_audit = audit_student(final, teacher_logits, labels)
    pd.DataFrame(
        [{"module_path": layer, "channel_index": channel, "group_id": gid}
         for layer, channel, gid in chosen]
    ).to_csv(OUT / f"{name}_selected_groups.csv", index=False)
    return final, realised, final_audit, teacher_logits, labels, teacher_audit

runs = {}
if RUN_SHALLOW:
    runs["shallow"] = run_lookahead("shallow", SHALLOW, shallow_scores, 4)
if RUN_DEEP:
    runs["deep"] = run_lookahead("deep", DEEP, deep_scores, 8)


In [ ]:
# One frozen minimal recovery epoch for deployed comparison.
train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
counts = np.bincount(train_y, minlength=len(CLASS_NAMES))
weights = np.zeros_like(counts, dtype=float)
weights[counts > 0] = 1.0 / np.sqrt(counts[counts > 0])
weights[counts > 0] /= weights[counts > 0].mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

g = torch.Generator().manual_seed(SEED)
subset_idx = torch.randperm(len(TRAIN_LOADER.dataset), generator=g)[:len(TRAIN_LOADER.dataset)//10]
MINIMAL_LOADER = DataLoader(
    Subset(TRAIN_LOADER.dataset, subset_idx.tolist()),
    batch_size=1024,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

def minimal_recovery(model):
    torch.manual_seed(SEED)
    model = model.to(DEVICE).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    for x, y in MINIMAL_LOADER:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    return model.eval()

summary = []
for name, (raw_model, realised, raw_audit, teacher_logits, labels, teacher_audit) in runs.items():
    recovered = minimal_recovery(deepcopy(raw_model))
    rec_audit = audit_student(recovered, teacher_logits, labels)
    checkpoint = OUT / f"{name}_asl_select_minimal_checkpoint.pt"
    torch.save({
        "state_dict": recovered.cpu().state_dict(),
        "architecture": name,
        "target_flops": TARGET_FLOPS,
        "realised_flops": realised,
        "class_names": CLASS_NAMES,
    }, checkpoint)
    recovered = recovered.to(DEVICE)
    summary.extend([
        {
            "architecture": name,
            "method": "asl_select",
            "regime": "raw",
            "realised_flops": realised,
            **{k: float(raw_audit[k]) for k in [
                "awbir", "fine_macro_f1", "family_macro_f1",
                "attack_to_benign_rate", "benign_to_attack_rate",
                "hsr_balanced_soc", "ece15",
            ]},
        },
        {
            "architecture": name,
            "method": "asl_select",
            "regime": "minimal",
            "realised_flops": realised,
            **{k: float(rec_audit[k]) for k in [
                "awbir", "fine_macro_f1", "family_macro_f1",
                "attack_to_benign_rate", "benign_to_attack_rate",
                "hsr_balanced_soc", "ece15",
            ]},
        },
    ])

summary = pd.DataFrame(summary)
summary.to_csv(OUT / "asl_select_summary.csv", index=False)
display(summary)


In [ ]:
# Compare only against already-frozen validation screens at the same architecture/regime.
gate_rows = []
if RUN_SHALLOW:
    baseline = pd.read_csv(REPO / "results/saber/19_recovery_regimes/recovery_regime_combined.csv")
    baseline = baseline[
        np.isclose(baseline["budget"], TARGET_FLOPS)
        & (baseline["regime"] == "minimal")
    ]
    asl = summary[(summary["architecture"] == "shallow") & (summary["regime"] == "minimal")].iloc[0]
    best_awbir = float(baseline["awbir"].min())
    best_family = float(baseline["family_macro_f1"].max())
    gate_rows.append({
        "architecture": "shallow",
        "asl_awbir": float(asl["awbir"]),
        "best_static_awbir": best_awbir,
        "asl_family_macro_f1": float(asl["family_macro_f1"]),
        "best_static_family_macro_f1": best_family,
        "awbir_win": bool(asl["awbir"] < best_awbir),
        "family_win": bool(asl["family_macro_f1"] > best_family),
        "attack_miss_safe": bool(
            asl["attack_to_benign_rate"]
            <= baseline["attack_to_benign_rate"].min() + 0.01
        ),
    })
if RUN_DEEP:
    baseline = pd.read_csv(REPO / "results/saber/20_depth_probe/depth_regime_results.csv")
    baseline = baseline[baseline["regime"] == "minimal"]
    asl = summary[(summary["architecture"] == "deep") & (summary["regime"] == "minimal")].iloc[0]
    gate_rows.append({
        "architecture": "deep",
        "asl_awbir": float(asl["awbir"]),
        "best_static_awbir": float(baseline["awbir"].min()),
        "asl_family_macro_f1": float(asl["family_macro_f1"]),
        "best_static_family_macro_f1": float(baseline["family_macro_f1"].max()),
        "awbir_win": bool(asl["awbir"] < baseline["awbir"].min()),
        "family_win": bool(asl["family_macro_f1"] > baseline["family_macro_f1"].max()),
        "attack_miss_safe": bool(
            asl["attack_to_benign_rate"]
            <= baseline["attack_to_benign_rate"].min() + 0.01
        ),
    })

gate_table = pd.DataFrame(gate_rows)
gate_table.to_csv(OUT / "ASL_selection_gate.csv", index=False)
passed = bool(
    len(gate_table)
    and (gate_table[["awbir_win", "family_win"]].sum(axis=1) >= 1).all()
    and gate_table["attack_miss_safe"].all()
)
gate = {
    "gate": "ASL_set_conditional_selection",
    "passed": passed,
    "criterion": (
        "At matched realised FLOPs after minimal recovery: beat the strongest "
        "static baseline on AWBIR or family macro-F1, without >0.01 excess "
        "attack-to-benign rate."
    ),
    "details": gate_rows,
}
(OUT / "ASL_selection_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
print(json.dumps(gate, indent=2))
